# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring a dataset using the `mlcroissant` library, referencing Croissant schema entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata as a Python object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for future referencing.

In [ ]:
# List all record sets and their fields by @id

print("Available record sets and fields (by @id):\n")
for recset in dataset.record_sets:
    print(f"RecordSet @id: {recset.id}")
    print(f"  Name: {recset.name}")
    print(f"  Description: {recset.description}")
    print(f"  Fields:")
    for field in recset.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
    print("\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Choose the primary survey record set by its @id (review previous cell output for actual @id)
# For demonstration, we will use the first available record set

record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

selected_record_set = record_set_ids[0]
print(f"Fields in record set {selected_record_set}:")
print(dataframes[selected_record_set].columns.tolist())
# Show sample records
dataframes[selected_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data for initial analysis. All references are made via the `@id` of relevant fields.

In [ ]:
# Pick a numeric field for demonstration (e.g., PHQ-9 total score or GAD-7 total score)
# Please replace these with their actual @id, which you can find above. We'll use the first numeric field found.
recset = [rs for rs in dataset.record_sets if rs.id == selected_record_set][0]
numeric_field = None
for field in recset.fields:
    if field.data_type in ('Integer', 'Float', 'Number'):
        numeric_field = field.id
        break

print(f"Selected numeric field @id: {numeric_field}")

# Filter for values greater than a threshold (example: 10)
threshold = 10
if numeric_field and numeric_field in dataframes[selected_record_set].columns:
    filtered_df = dataframes[selected_record_set][dataframes[selected_record_set][numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Group by a categorical field (choose the first string/categorical field found)
    group_field = None
    for field in recset.fields:
        if field.data_type in ('Text', 'String') and field.id != numeric_field and field.id in filtered_df.columns:
            group_field = field.id
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in dataframes[selected_record_set].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[selected_record_set][numeric_field].dropna(), bins=16, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=dataframes[selected_record_set][group_field], y=dataframes[selected_record_set][numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze a Croissant-encoded mental health survey dataset using the `mlcroissant` library.

- All entities (record sets, fields) were referenced by their `@id` for transparency and schema consistency.
- We performed basic EDA, including filtering by a numeric field, normalization, and grouping by categorical variables.
- Visualizations highlighted data distributions and possible group differences.

You are encouraged to further investigate the dataset, explore more fields, and customize analyses for your research.